In [ ]:
from transformers import pipeline

model = pipeline("sentiment-analysis")
print(model("I love this project"))

In [ ]:
import requests
from datetime import datetime, timedelta

API_KEY = "b2a0d07341544104b699c5576a1bf7fd"
url = "https://newsapi.org/v2/everything"

# Query parameters
query = "finance AND (stock market OR investment)"
language = "en"
page_size = 100  # max per request

# Date range setup: last 30 days
end_date = datetime.today()
start_date = end_date - timedelta(days=30)

all_articles = []

# Split the 30-day range into 3-day chunks (adjust as needed)
delta = timedelta(days=3)
current_start = start_date

while current_start < end_date:
    current_end = min(current_start + delta, end_date)
    
    page = 1
    while True:
        params = {
            "q": query,
            "language": language,
            "pageSize": page_size,
            "page": page,
            "from": current_start.strftime("%Y-%m-%d"),
            "to": current_end.strftime("%Y-%m-%d"),
            "apiKey": API_KEY
        }

        response = requests.get(url, params=params)
        data = response.json()

        if "articles" in data and data["articles"]:
            all_articles.extend(data["articles"])
            if len(data["articles"]) < page_size:
                break  # No more pages for this date range
            page += 1
        else:
            break  # No articles for this page/date range

    current_start += delta

# Extract text
news_list = []
for article in all_articles:
    text = article.get("title", "") + " " + str(article.get("description", ""))
    news_list.append(text)

print(f"Total articles fetched: {len(news_list)}")
print(news_list[:10])

In [ ]:
# Assuming 'results' is the output of sentiment_model(news_list)
# Map scores to labels
score_map = {"POSITIVE": "Bullish", "NEGATIVE": "Bearish"}

# Count occurrences
sentiment_counts = {"Bullish": 0, "Bearish": 0, "Neutral": 0}

for r in results:
    label = r['label']
    score = r['score']
    
    # Optional: set a threshold for Neutral
    if score < 0.6:
        sentiment_counts["Neutral"] += 1
    else:
        sentiment_counts[score_map[label]] += 1

print(sentiment_counts)

In [ ]:
import matplotlib.pyplot as plt

labels = list(sentiment_counts.keys())
counts = list(sentiment_counts.values())

plt.figure(figsize=(6,4))
plt.bar(labels, counts, color=['green', 'red', 'gray'])
plt.title("Market Sentiment Distribution")
plt.xlabel("Sentiment")
plt.ylabel("Number of Articles")
plt.show()